# MOONPIERCER 
## Exhaustive PBH Lunar Crater Search Analysis

Publication-grade analysis of the MOONPIERCER chord-search pipeline results.
Loads HPC outputs, performs multi-hypothesis rescoring, sensitivity sweeps,
and derives dark-matter fraction constraints.

**Question:** *Are there two craters on the Moon that could have been caused
by the transit of a primordial black hole?*

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from moonpiercer.config import ChordConfig
from moonpiercer.constants import (
    LUNAR_RADIUS_M,
    LUNAR_SURFACE_AREA_KM2,
    KAPPA_CLASSICAL_LOW_M2_PER_MYR,
    KAPPA_CLASSICAL_HIGH_M2_PER_MYR,
    KAPPA_ADOPTED_M2_PER_MYR,
    DM_FLUX_HALO_G_PER_S,
)
from moonpiercer.geometry import (
    chord_length_from_separation,
    expected_ellipticity_from_separation,
    lonlat_to_unit_vectors,
)
from moonpiercer.velocity import (
    maxwell_boltzmann_speed_pdf,
    rotation_offset_deg,
    max_physical_angular_offset_deg,
)
from moonpiercer.plotting import (
    plot_spatial_coverage,
    plot_score_distribution,
    plot_chord_map,
    plot_score_component_star,
    plot_null_distribution,
    plot_pair_scores_with_threshold,
    plot_crater_map_with_pairs,
)
from moonpiercer.pairing import rescore_pairs
from moonpiercer.null_model import (
    compute_significance,
    benjamini_hochberg,
    empirical_p_value,
    percentile_score,
)
from moonpiercer.io_utils import (
    save_figure,
    save_dataframe,
    load_dataframe,
    load_json,
    RESULTS_DIR,
    PAPER_FIGURES_DIR,
    ensure_dir,
)

%matplotlib inline

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 8,
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "axes.linewidth": 0.6,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.minor.size": 1.5,
    "ytick.minor.size": 1.5,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.fontsize": 7,
    "legend.framealpha": 0.9,
    "figure.constrained_layout.use": True,
})

ensure_dir(PAPER_FIGURES_DIR)

COL_W = 3.54
FULL_W = 7.25

cfg = ChordConfig()
print("Setup complete.")

## 1. Load HPC Results

In [ ]:
RUN_NAME = "moonpiercer_full_run"

def _select_run_root(results_dir: Path, run_name: str) -> Path:
    candidate = results_dir / run_name
    if candidate.exists():
        return candidate
    dirs = sorted(
        [d for d in results_dir.iterdir() if d.is_dir() and "moonpiercer" in d.name],
        key=lambda d: d.stat().st_mtime, reverse=True,
    )
    return dirs[0] if dirs else candidate

run_root = _select_run_root(RESULTS_DIR, RUN_NAME)
chip_dir = run_root / "chips"
global_dir = run_root / "global"

print(f"Run directory: {run_root}")
print(f"  exists: {run_root.exists()}")

craters = pd.DataFrame()
HAS_CHIPS = False
csv_files = []
if chip_dir.exists():
    csv_files = sorted(chip_dir.glob("*/craters.csv"))
    if csv_files:
        frames = [pd.read_csv(f) for f in csv_files if f.stat().st_size > 0]
        if frames:
            craters = pd.concat(frames, ignore_index=True)
            HAS_CHIPS = True
print(f"Craters loaded: {len(craters):,}  (from {len(csv_files)} chip CSVs)")

all_pairs = load_dataframe(global_dir / "all_pairs_scored.csv")
sig_pairs = load_dataframe(global_dir / "significant_pairs.csv")
null_scores = np.array([])
null_path = global_dir / "null_best_scores.npy"
if null_path.exists():
    null_scores = np.load(str(null_path))

null_pair_details: list[dict] = []
null_details_path = global_dir / "null_pair_details.json"
if null_details_path.exists():
    null_pair_details = load_json(null_details_path)

summary_path = global_dir / "run_summary.json"
run_summary = load_json(summary_path) if summary_path.exists() else {}

HAS_RESULTS = len(all_pairs) > 0 and len(null_scores) > 0

print(f"Pairs loaded:  {len(all_pairs):,}")
print(f"Significant:   {len(sig_pairs):,}")
print(f"Null trials:   {len(null_scores):,}")
print(f"Null details:  {len(null_pair_details):,}")

## 2. Survey Overview

In [ ]:
if run_summary:
    print("Survey Summary")
    print("=" * 50)
    for key in ["n_chips_discovered", "n_chips_with_data", "n_craters",
                "coverage_km2", "n_raw_pairs", "n_top_pairs",
                "n_significant", "best_score", "best_p_value",
                "fdr_alpha", "random_trials"]:
        val = run_summary.get(key)
        if val is not None:
            if isinstance(val, float):
                print(f"  {key:30s} {val:>14.4f}")
            elif isinstance(val, int):
                print(f"  {key:30s} {val:>14,}")
            else:
                print(f"  {key:30s} {val}")
    cov_frac = run_summary.get("coverage_km2", 0) / LUNAR_SURFACE_AREA_KM2
    print(f"  {'coverage_fraction':30s} {cov_frac:>14.6f}")
else:
    print("No run summary found.")

In [ ]:
if HAS_CHIPS and not craters.empty:
    fig_cov = plot_spatial_coverage(craters, figsize=(FULL_W, FULL_W * 0.5))
    save_figure(fig_cov, "spatial_coverage", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()
else:
    print("No chip data for spatial coverage plot.")

In [ ]:
if HAS_CHIPS and chip_dir.exists():
    chip_dirs = sorted(d for d in chip_dir.iterdir() if d.is_dir())
    n_total = len(chip_dirs)
    n_with_craters = sum(
        1 for d in chip_dirs
        if (d / "craters.csv").exists() and (d / "craters.csv").stat().st_size > 10
    )
    n_empty = n_total - n_with_craters
    print("Chip health breakdown:")
    print(f"  Total chip directories:   {n_total:,}")
    print(f"  With crater detections:   {n_with_craters:,} ({100*n_with_craters/max(n_total,1):.1f}%)")
    print(f"  Empty / failed:           {n_empty:,} ({100*n_empty/max(n_total,1):.1f}%)")
else:
    print("No chip directory found.")

## 3. Crater Population Analysis

In [ ]:
if not HAS_CHIPS:
    print("Skipping: no crater data.")
else:
    print(f"Total craters: {len(craters):,}")
    print(f"\\nRadius statistics [m]:")
    print(craters["radius_m"].describe().to_string())
    if "freshness_index" in craters.columns:
        print(f"\\nFreshness Index statistics:")
        print(craters["freshness_index"].describe().to_string())
        n_above_fi = (craters["freshness_index"] >= cfg.min_freshness).sum()
        print(f"\\nCraters above FI threshold ({cfg.min_freshness}): "
              f"{n_above_fi:,} ({100*n_above_fi/len(craters):.1f}%)")
    if "depth_proxy" in craters.columns:
        n_above_dp = (craters["depth_proxy"] >= cfg.min_depth_proxy).sum()
        print(f"Craters above depth proxy threshold ({cfg.min_depth_proxy}): "
              f"{n_above_dp:,} ({100*n_above_dp/len(craters):.1f}%)")

In [ ]:
if HAS_CHIPS:
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
    ax.hist(craters["radius_m"], bins=60, range=(0, 15), color="0.4",
            edgecolor="0.2", linewidth=0.3)
    ax.axvline(cfg.target_crater_radius_range_m[0], color="C3", ls="--", lw=0.8,
               label=f"Detection range [{cfg.target_crater_radius_range_m[0]}, "
                     f"{cfg.target_crater_radius_range_m[1]}] m")
    ax.axvline(cfg.target_crater_radius_range_m[1], color="C3", ls="--", lw=0.8)
    ax.set_xlabel("Crater radius [m]")
    ax.set_ylabel("Count")
    ax.legend()
    save_figure(fig, "crater_radius_distribution", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_CHIPS and "freshness_index" in craters.columns:
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
    ax.hist(craters["freshness_index"].dropna(), bins=80, range=(0, 1),
            color="steelblue", edgecolor="0.2", linewidth=0.3)
    ax.axvline(cfg.min_freshness, color="C3", ls="--", lw=0.8,
               label=f"Pairing threshold (FI >= {cfg.min_freshness})")
    ax.set_xlabel("Freshness Index")
    ax.set_ylabel("Count")
    ax.legend()
    save_figure(fig, "freshness_distribution", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_CHIPS and "lat_deg" in craters.columns:
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
    lat_bins = np.arange(-90, 91, 10)
    lat_centres = 0.5 * (lat_bins[:-1] + lat_bins[1:])
    counts, _ = np.histogram(craters["lat_deg"], bins=lat_bins)
    solid_angle = np.abs(
        np.sin(np.radians(lat_bins[1:])) - np.sin(np.radians(lat_bins[:-1]))
    )
    density = counts / solid_angle
    density = density / density.max()
    ax.bar(lat_centres, density, width=9, color="0.5", edgecolor="0.2", linewidth=0.3)
    ax.set_xlabel("Selenographic latitude [deg]")
    ax.set_ylabel("Relative crater density")
    ax.set_xlim(-90, 90)
    save_figure(fig, "latitude_density_profile", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_CHIPS and "shape_reliable" in craters.columns:
    n_reliable = int(craters["shape_reliable"].sum())
    n_unreliable = len(craters) - n_reliable
    print(f"Shape-reliable craters: {n_reliable:,} ({100*n_reliable/len(craters):.1f}%)")
    print(f"Shape-unreliable:       {n_unreliable:,} ({100*n_unreliable/len(craters):.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, COL_W * 0.6))
    axes[0].pie([n_reliable, n_unreliable], labels=["Reliable", "Unreliable"],
                colors=["C0", "0.7"], autopct="%1.1f%%", startangle=90,
                textprops={"fontsize": 7})
    axes[0].set_title("Shape reliability", fontsize=8)

    if "ellipticity" in craters.columns:
        reliable = craters.loc[craters["shape_reliable"], "ellipticity"].dropna()
        axes[1].hist(reliable, bins=60, range=(1, 5), color="C0",
                     edgecolor="0.2", linewidth=0.3)
        axes[1].set_xlabel("Ellipticity")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Ellipticity (shape-reliable)", fontsize=8)
    save_figure(fig, "shape_reliability", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_CHIPS and "depth_proxy" in craters.columns:
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
    sample = craters.sample(min(20_000, len(craters)), random_state=42)
    ax.scatter(sample["radius_m"], sample["depth_proxy"],
               s=0.3, alpha=0.15, color="0.3", rasterized=True)
    ax.axhline(cfg.min_depth_proxy, color="C3", ls="--", lw=0.8,
               label=f"Threshold = {cfg.min_depth_proxy}")
    rho, pval = stats.spearmanr(craters["radius_m"], craters["depth_proxy"])
    ax.text(0.95, 0.95, f"Spearman $\\\\rho$ = {rho:.3f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=7)
    ax.set_xlabel("Crater radius [m]")
    ax.set_ylabel("Depth proxy")
    ax.legend()
    save_figure(fig, "depth_proxy_vs_radius", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

## 4. Baseline Pair Analysis (Default Config)

In [ ]:
if not HAS_RESULTS:
    print("Skipping: no pair/null data.")
else:
    print(f"Total scored pairs:     {len(all_pairs):,}")
    n_sig = len(sig_pairs)
    print(f"Significant (BH-FDR):   {n_sig}")
    if n_sig == 0:
        print("  No statistically significant detections after BH-FDR correction.")

    score_cols = [c for c in all_pairs.columns if c.startswith("T_")]
    display_cols = ["score", "separation_deg", "radius_a_m", "radius_b_m",
                    "fi_a", "fi_b"] + score_cols
    if "p_value" in all_pairs.columns:
        display_cols.append("p_value")
    avail = [c for c in display_cols if c in all_pairs.columns]
    n_show = min(10, len(all_pairs))
    print(f"\\nTop {n_show} pairs:")
    print(all_pairs[avail].head(n_show).to_string(index=False, float_format="{:.4f}".format))

In [ ]:
if HAS_RESULTS and len(all_pairs) > 0:
    fig_star = plot_score_component_star(all_pairs, n_top=10, figsize=(COL_W, COL_W))
    save_figure(fig_star, "score_component_star", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_RESULTS:
    best_real = float(all_pairs["score"].iloc[0])
    fig_null = plot_null_distribution(
        null_scores, best_real_score=best_real, figsize=(COL_W, COL_W * 0.75))
    save_figure(fig_null, "null_distribution_baseline", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

    q95 = np.percentile(null_scores, 95)
    q99 = np.percentile(null_scores, 99)
    print(f"Null 95th percentile: {q95:.6f}")
    print(f"Null 99th percentile: {q99:.6f}")
    print(f"Null mean:            {null_scores.mean():.6f}")
    print(f"Null std:             {null_scores.std():.6f}")
    print(f"Best real score:      {best_real:.6f}")
    print(f"Best p-value:         {empirical_p_value(best_real, null_scores):.6f}")

### 4b. Percentile-Rank Scores

The raw composite score (product of Gaussian terms) clusters near 1 and is
hard to interpret visually.  The **percentile score** maps each pair to
[0, 1] by measuring what fraction of null-trial best scores it beats.
A value of 0.95 means the pair outscores 95% of random catalogues.

All downstream analysis uses `percentile_score` as the default ranking.

In [ ]:
# Compute percentile scores (or use pre-computed column from HPC run)
if HAS_RESULTS:
    if "percentile_score" not in all_pairs.columns:
        all_pairs["percentile_score"] = np.array([
            percentile_score(s, null_scores)
            for s in all_pairs["score"].values
        ])
        print("Computed percentile_score on the fly (not in CSV).")
    else:
        print("Using pre-computed percentile_score from HPC output.")

    ps = all_pairs["percentile_score"].values
    print(f"\nPercentile score statistics:")
    print(f"  min:    {ps.min():.4f}")
    print(f"  median: {np.median(ps):.4f}")
    print(f"  mean:   {ps.mean():.4f}")
    print(f"  max:    {ps.max():.4f}")
    print(f"  Pairs above 0.95: {(ps >= 0.95).sum()}")
    print(f"  Pairs above 0.99: {(ps >= 0.99).sum()}")

In [ ]:
# Percentile score distribution
if HAS_RESULTS:
    fig, axes = plt.subplots(1, 2, figsize=(FULL_W, COL_W * 0.75))

    # Left: histogram of percentile scores (all pairs)
    ax = axes[0]
    ax.hist(all_pairs["percentile_score"], bins=50, range=(0, 1),
            color="steelblue", edgecolor="0.2", linewidth=0.3)
    ax.axvline(0.95, color="C3", ls="--", lw=0.8, label="95th percentile")
    ax.set_xlabel("Percentile score")
    ax.set_ylabel("Number of pairs")
    ax.set_title("Distribution of percentile scores", fontsize=8)
    ax.legend(fontsize=6)

    # Right: raw score vs percentile score scatter
    ax = axes[1]
    ax.scatter(all_pairs["score"], all_pairs["percentile_score"],
               s=1.5, alpha=0.4, color="0.3", rasterized=True)
    ax.set_xlabel("Raw composite score")
    ax.set_ylabel("Percentile score")
    ax.set_title("Raw vs percentile score", fontsize=8)
    ax.axhline(0.95, color="C3", ls="--", lw=0.6, alpha=0.7)

    save_figure(fig, "percentile_score_distribution", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
# Top pairs ranked by percentile score
if HAS_RESULTS:
    top_by_pct = all_pairs.sort_values("percentile_score", ascending=False)
    display_cols = ["percentile_score", "score", "separation_deg",
                    "radius_a_m", "radius_b_m", "fi_a", "fi_b"]
    if "p_value" in top_by_pct.columns:
        display_cols.append("p_value")
    avail = [c for c in display_cols if c in top_by_pct.columns]
    n_show = min(15, len(top_by_pct))
    print(f"Top {n_show} pairs by percentile score:")
    print(top_by_pct[avail].head(n_show).to_string(
        index=False, float_format="{:.4f}".format))

In [ ]:
if HAS_RESULTS:
    real_scores = all_pairs["score"].values
    fig_thresh = plot_pair_scores_with_threshold(
        real_scores, null_scores, figsize=(COL_W, COL_W * 0.75))
    save_figure(fig_thresh, "pair_scores_with_threshold", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_RESULTS and len(all_pairs) >= 2:
    fig_chords = plot_chord_map(all_pairs, n_best=20, figsize=(FULL_W, FULL_W * 0.5))
    save_figure(fig_chords, "chord_map_top20", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_RESULTS and HAS_CHIPS and len(all_pairs) >= 2:
    fig_cmap = plot_crater_map_with_pairs(
        craters, all_pairs, n_best=10, figsize=(FULL_W, FULL_W * 0.5))
    save_figure(fig_cmap, "crater_map_with_pairs", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

## 5. Multi-Hypothesis Scoring Rounds

Rescore real pairs **and** null-trial best pairs under alternative scoring
hypotheses to test the robustness of the (non-)detection.

| Round | Hypothesis | Config overrides |
|-------|-----------|-----------------|
| H0 | Full scoring (default) | -- |
| H1 | Size may differ | sigma_radius=1e6 |
| H2 | Shape unreliable | sigma_ellipticity=1e6, sigma_orientation_deg=1e6 |
| H3 | Position prediction unreliable | sigma_position_deg=1e6 |
| H4 | Only freshness + geometry (H1+H2+H3) | all three relaxed |
| H5 | Conservative: maximum penalty | tightest sigmas |

In [ ]:
if not HAS_RESULTS or len(all_pairs) == 0:
    print("Skipping: no pair data.")
else:
    HYPOTHESES = {
        "H0": {"label": "Full scoring (default)", "overrides": {}},
        "H1": {"label": "Radius relaxed",
               "overrides": {"sigma_radius": 1e6}},
        "H2": {"label": "Shape relaxed",
               "overrides": {"sigma_ellipticity": 1e6, "sigma_orientation_deg": 1e6}},
        "H3": {"label": "Position relaxed",
               "overrides": {"sigma_position_deg": 1e6}},
        "H4": {"label": "Freshness + geometry only",
               "overrides": {"sigma_radius": 1e6, "sigma_ellipticity": 1e6,
                             "sigma_orientation_deg": 1e6, "sigma_position_deg": 1e6}},
        "H5": {"label": "Conservative (tight)",
               "overrides": {"sigma_freshness": 0.005, "sigma_ellipticity": 0.02,
                             "sigma_orientation_deg": 1.5, "sigma_position_deg": 0.25}},
    }

    round_results = {}
    has_null_details = len(null_pair_details) > 0
    null_details_df = pd.DataFrame(null_pair_details) if has_null_details else pd.DataFrame()

    for hname, hinfo in HYPOTHESES.items():
        custom_cfg = ChordConfig(**hinfo["overrides"])

        rescored = rescore_pairs(all_pairs, config=custom_cfg)
        rescored = rescored.sort_values("score", ascending=False).reset_index(drop=True)

        if has_null_details and not null_details_df.empty:
            null_rescored = rescore_pairs(null_details_df, config=custom_cfg)
            if "trial" in null_rescored.columns:
                null_best = null_rescored.groupby("trial")["score"].max().values
            else:
                null_best = null_scores
        else:
            null_best = null_scores

        scored = compute_significance(rescored, null_best, alpha=cfg.fdr_alpha)
        n_sig_h = int(scored["bh_significant"].sum()) if "bh_significant" in scored.columns else 0
        best_score_h = float(scored["score"].iloc[0]) if len(scored) > 0 else 0.0
        best_pct_h = float(scored["percentile_score"].iloc[0]) if "percentile_score" in scored.columns and len(scored) > 0 else 0.0
        best_p_h = float(scored["p_value"].iloc[0]) if "p_value" in scored.columns and len(scored) > 0 else 1.0

        if hname == "H0":
            h0_ranking = rescored["score"].values
        tau_val = 1.0
        if hname != "H0" and len(rescored) >= 2:
            merged = all_pairs[["score"]].rename(columns={"score": "score_h0"}).copy()
            merged["score_hx"] = rescored["score"].values[:len(merged)]
            tau_val, _ = stats.kendalltau(merged["score_h0"], merged["score_hx"])

        round_results[hname] = {
            "label": hinfo["label"],
            "best_score": best_score_h,
            "best_pct": best_pct_h,
            "best_p": best_p_h,
            "n_significant": n_sig_h,
            "kendall_tau_vs_h0": tau_val,
            "null_best": null_best,
            "scored": scored,
        }

    print(f"{'Round':<6} {'Hypothesis':<28} {'Best Pct':>9} {'Best Raw':>10} {'p-value':>9} {'N_sig':>6} {'tau H0':>8}")
    print("-" * 80)
    for hname, r in round_results.items():
        print(f"{hname:<6} {r['label']:<28} {r['best_pct']:>9.4f} {r['best_score']:>10.6f} "
              f"{r['best_p']:>9.4f} {r['n_significant']:>6d} {r['kendall_tau_vs_h0']:>8.4f}")

In [ ]:
if HAS_RESULTS and 'round_results' in dir() and round_results:
    n_rounds = len(round_results)
    fig, axes = plt.subplots(n_rounds, 1, figsize=(COL_W, COL_W * 0.45 * n_rounds), sharex=False)
    if n_rounds == 1:
        axes = [axes]

    for ax, (hname, r) in zip(axes, round_results.items()):
        null_b = r["null_best"]
        ax.hist(null_b, bins=50, color="0.6", edgecolor="0.3", linewidth=0.3,
                density=True, label="Null")
        ax.axvline(r["best_score"], color="C3", ls="-", lw=1.0,
                   label=f"Best real = {r['best_score']:.4f}")
        ax.set_ylabel("Density", fontsize=7)
        ax.set_title(f"{hname}: {r['label']}", fontsize=7, loc="left")
        ax.legend(fontsize=6)

    axes[-1].set_xlabel("Best pair score")
    save_figure(fig, "multi_hypothesis_null_distributions", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_RESULTS and 'round_results' in dir() and len(round_results) > 1:
    labels = list(round_results.keys())
    n = len(labels)
    tau_matrix = np.ones((n, n))
    for i, hi in enumerate(labels):
        for j, hj in enumerate(labels):
            if i != j:
                si = round_results[hi]["scored"]["score"].values
                sj = round_results[hj]["scored"]["score"].values
                length = min(len(si), len(sj))
                tau_matrix[i, j], _ = stats.kendalltau(si[:length], sj[:length])

    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.85))
    im = ax.imshow(tau_matrix, vmin=-1, vmax=1, cmap="RdBu_r", aspect="equal")
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, fontsize=7, rotation=45, ha="right")
    ax.set_yticklabels(labels, fontsize=7)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{tau_matrix[i,j]:.2f}", ha="center", va="center", fontsize=6)
    fig.colorbar(im, ax=ax, label=r"Kendall $\\tau$", shrink=0.8)
    ax.set_title("Rank correlation between scoring hypotheses", fontsize=8)
    save_figure(fig, "kendall_tau_heatmap", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

## 6. Sensitivity Analysis (Sigma Sweeps)

Sweep each active scoring sigma from 0.3x to 3.0x its default value and
track the best pair score, rank stability (Kendall tau), and number of
pairs above the null 95th percentile.

In [ ]:
if not HAS_RESULTS or len(all_pairs) == 0:
    print("Skipping: no pair data.")
else:
    SIGMA_PARAMS = {
        "sigma_freshness": cfg.sigma_freshness,
        "sigma_radius": cfg.sigma_radius,
        "sigma_ellipticity": cfg.sigma_ellipticity,
        "sigma_orientation_deg": cfg.sigma_orientation_deg,
        "sigma_position_deg": cfg.sigma_position_deg,
    }

    scale_factors = np.linspace(0.3, 3.0, 60)
    null_95 = np.percentile(null_scores, 95)
    baseline_scores = all_pairs["score"].values
    top_k_tau = min(10, len(all_pairs))

    sweep_results = {}

    for param_name, default_val in SIGMA_PARAMS.items():
        best_scores = []
        tau_vals = []
        n_above_null = []

        for sf in scale_factors:
            c = ChordConfig(**{param_name: default_val * sf})
            rescored = rescore_pairs(all_pairs, config=c)
            rescored = rescored.sort_values("score", ascending=False).reset_index(drop=True)

            best_scores.append(float(rescored["score"].iloc[0]))
            n_above_null.append(int((rescored["score"] >= null_95).sum()))

            tau, _ = stats.kendalltau(
                baseline_scores[:top_k_tau],
                rescored["score"].values[:top_k_tau],
            )
            tau_vals.append(tau)

        sweep_results[param_name] = {
            "best_scores": np.array(best_scores),
            "tau_vals": np.array(tau_vals),
            "n_above_null": np.array(n_above_null),
            "default_val": default_val,
        }

    print("Sigma sweeps complete.")

In [ ]:
if HAS_RESULTS and 'sweep_results' in dir() and sweep_results:
    n_params = len(sweep_results)
    fig, axes = plt.subplots(n_params, 1, figsize=(COL_W, COL_W * 0.5 * n_params), sharex=True)
    if n_params == 1:
        axes = [axes]

    for ax, (pname, sr) in zip(axes, sweep_results.items()):
        ax.plot(scale_factors, sr["best_scores"], "k-", lw=0.8)
        ax.axvline(1.0, color="C3", ls="--", lw=0.6, alpha=0.7)
        ax.set_ylabel("Best score", fontsize=7)
        ax.set_title(pname.replace("_", " "), fontsize=7, loc="left")

    axes[-1].set_xlabel("Scale factor")
    save_figure(fig, "sensitivity_best_score", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

In [ ]:
if HAS_RESULTS and 'sweep_results' in dir() and sweep_results:
    fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.75))
    for i, (pname, sr) in enumerate(sweep_results.items()):
        short = pname.replace("sigma_", "").replace("_deg", "")
        ax.plot(scale_factors, sr["tau_vals"], lw=0.8, label=short, color=f"C{i}")
    ax.axvline(1.0, color="0.5", ls=":", lw=0.5)
    ax.set_xlabel("Scale factor")
    ax.set_ylabel(r"Kendall $\\tau$ (top-10 vs baseline)")
    ax.legend(fontsize=6)
    ax.set_ylim(-0.1, 1.05)
    save_figure(fig, "sensitivity_kendall_tau", pdf_dir=PAPER_FIGURES_DIR)
    plt.show()

## 7. Dark-Matter Fraction Constraint ($f_{\rm PBH}$ Upper Limit)

For a non-detection (0 observed PBH transits), the Poisson 95% CL upper
limit on the expected number of events is $N_{\rm exp} < 3.0$.

$$
f_{\rm PBH}^{95\%} = \frac{3.0 \, m_{\rm PBH}}
    {\Phi_{\rm DM} \, \sigma_{\rm geom} \, \tau_{\rm surv} \, f_{\rm cov}}
$$

where $\Phi_{\rm DM}$ is the DM mass flux, $\sigma_{\rm geom} = \pi R_{\rm Moon}^2$,
$\tau_{\rm surv} = r_{\rm eff}^2 / (4\kappa)$, and $f_{\rm cov}$ is the
survey coverage fraction.

In [ ]:
R_moon_cm = LUNAR_RADIUS_M * 100
sigma_geom_cm2 = np.pi * R_moon_cm**2
phi_dm = DM_FLUX_HALO_G_PER_S

coverage_km2 = run_summary.get("coverage_km2", 0.0) if run_summary else 0.0
f_cov = coverage_km2 / LUNAR_SURFACE_AREA_KM2 if coverage_km2 > 0 else 1e-4
print(f"Survey coverage: {coverage_km2:.1f} km^2 / {LUNAR_SURFACE_AREA_KM2:.1f} km^2 = {f_cov:.6f}")

if HAS_CHIPS and "radius_m" in craters.columns:
    r_eff_m = float(np.sqrt(np.mean(craters["radius_m"]**2)))
else:
    r_eff_m = 3.0
print(f"Effective crater radius: {r_eff_m:.2f} m")

s_per_myr = 1e6 * 365.25 * 24 * 3600
tau_surv_adopted_s = (r_eff_m**2 / (4 * KAPPA_ADOPTED_M2_PER_MYR)) * s_per_myr
tau_surv_low_s = (r_eff_m**2 / (4 * KAPPA_CLASSICAL_LOW_M2_PER_MYR)) * s_per_myr
tau_surv_high_s = (r_eff_m**2 / (4 * KAPPA_CLASSICAL_HIGH_M2_PER_MYR)) * s_per_myr
tau_surv_anom_s = tau_surv_adopted_s * 56  # anomalous diffusion (Fassett 2022)

print(f"tau_surv (adopted kappa): {tau_surv_adopted_s / s_per_myr:.3f} Myr")
print(f"tau_surv (anomalous):     {tau_surv_anom_s / s_per_myr:.3f} Myr")

m_pbh_g = np.logspace(14, 22, 500)
N_UL = 3.0

def f_pbh_upper(m_g, tau_s, f_coverage):
    return N_UL * m_g / (phi_dm * sigma_geom_cm2 * tau_s * f_coverage)

f_adopted = f_pbh_upper(m_pbh_g, tau_surv_adopted_s, f_cov)
f_low_kappa = f_pbh_upper(m_pbh_g, tau_surv_low_s, f_cov)
f_high_kappa = f_pbh_upper(m_pbh_g, tau_surv_high_s, f_cov)
f_anom = f_pbh_upper(m_pbh_g, tau_surv_anom_s, f_cov)
f_adopted_full = f_pbh_upper(m_pbh_g, tau_surv_adopted_s, 1.0)

idx_18 = np.argmin(np.abs(m_pbh_g - 1e18))
print(f"\\nAt m_PBH = 1e18 g:")
print(f"  f_PBH (adopted kappa): {f_adopted[idx_18]:.2e}")
print(f"  f_PBH (anomalous):     {f_anom[idx_18]:.2e}")
print(f"  f_PBH (full coverage): {f_adopted_full[idx_18]:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(COL_W, COL_W * 0.9))

ax.fill_between(m_pbh_g, f_high_kappa, f_low_kappa,
                alpha=0.15, color="C0", label=r"Classical $\\kappa$ band")
ax.plot(m_pbh_g, f_adopted, "C0-", lw=1.0,
        label=r"Adopted $\\kappa$ = 1.6 m$^2$ Myr$^{-1}$")
ax.plot(m_pbh_g, f_anom, "C1--", lw=0.8,
        label=r"Anomalous diffusion ($56\\times$)")
ax.plot(m_pbh_g, f_adopted_full, "C2:", lw=0.8,
        label=r"Projected ($f_{\\rm cov} = 1$)")

ax.axhline(1.0, color="0.3", ls="-", lw=0.5)
ax.text(1e14, 1.3, r"$f_{\\rm PBH} = 1$", fontsize=6, color="0.3")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$m_{\\rm PBH}$ [g]")
ax.set_ylabel(r"$f_{\\rm PBH}$ (95% CL upper limit)")
ax.set_xlim(1e14, 1e22)
ax.set_ylim(1e-6, 1e6)
ax.legend(fontsize=6, loc="upper left")

save_figure(fig, "f_pbh_constraint", pdf_dir=PAPER_FIGURES_DIR)
plt.show()

## 8. Results Summary

In [ ]:
print("=" * 70)
print("MOONPIERCER -- Results Summary")
print("=" * 70)

if run_summary:
    print(f"\nSurvey: {run_summary.get('n_chips_with_data', '?')} chips, "
          f"{run_summary.get('n_craters', '?'):,} craters, "
          f"{run_summary.get('coverage_km2', 0):.0f} km^2 coverage")
    print(f"Null model: {run_summary.get('random_trials', '?')} MC trials")

if HAS_RESULTS:
    best = all_pairs.iloc[0]
    print(f"\nBest pair (raw score):        {best['score']:.6f}")
    if "percentile_score" in best.index:
        print(f"Best pair (percentile score): {best['percentile_score']:.4f}")
    if "p_value" in best.index:
        print(f"Best pair (p-value):          {best['p_value']:.6f}")
    print(f"Significant pairs (BH-FDR at alpha={cfg.fdr_alpha}): {len(sig_pairs)}")

    print("\nMulti-hypothesis robustness:")
    if 'round_results' in dir():
        all_nsig = [r["n_significant"] for r in round_results.values()]
        all_pcts = [r["best_pct"] for r in round_results.values()]
        print(f"  Significant detections across {len(round_results)} hypotheses: {all_nsig}")
        print(f"  Best percentile scores: {[f'{p:.4f}' for p in all_pcts]}")
        if max(all_nsig) == 0:
            print("  --> No scoring hypothesis yields a significant detection.")

print("\nDM fraction constraint (m_PBH = 1e18 g):")
if 'f_adopted' in dir():
    print(f"  f_PBH < {f_adopted[idx_18]:.2e} (adopted kappa, current coverage)")
    print(f"  f_PBH < {f_adopted_full[idx_18]:.2e} (adopted kappa, full coverage)")

print("\n" + "=" * 70)
print("Under no scoring hypothesis do we find statistically significant")
print("evidence for PBH-transit crater pairs in the current survey.")
print("The derived f_PBH constraints are shown above.")
print("=" * 70)